In [6]:

import json
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel


# =====================================
# 1. Initialize Gemini
# =====================================

load_dotenv()
client = genai.Client()


# =====================================
# 2. Define Pydantic Models
# =====================================

class ActionItem(BaseModel):
    task: str
    owner: str


class ActionItemList(BaseModel):
    action_items: list[ActionItem]


# =====================================
# 3. Set File Paths
# =====================================

# meeting.txt must be in the same folder
# where your Jupyter Notebook is running.

PROJECT_DIR = Path.cwd()

MEETING_FILE = PROJECT_DIR / "meeting.txt"
TEXT_OUTPUT = PROJECT_DIR / "action_items.txt"
JSON_OUTPUT = PROJECT_DIR / "action_items.json"

print("Current folder:", PROJECT_DIR)


# =====================================
# 4. Read Meeting Notes
# =====================================

if not MEETING_FILE.exists():

    print("\nERROR: meeting.txt not found!")
    print("Please put meeting.txt inside this folder:")
    print(PROJECT_DIR)

    print("\nFiles currently available:")
    for item in PROJECT_DIR.iterdir():
        print("-", item.name)

    raise FileNotFoundError("Please place meeting.txt in the notebook folder.")

with open(MEETING_FILE, "r", encoding="utf-8") as file:
    meeting_notes = file.read()

if not meeting_notes.strip():
    raise ValueError("Meeting notes file is empty.")

print("\nMeeting notes loaded successfully!")


# =====================================
# 5. Prepare Prompt
# =====================================

prompt = f"""
Extract all actionable tasks from these meeting notes.

For every task identify:
- task: What needs to be done?
- owner: Who is responsible?

Rules:
- Do not invent tasks or owners.
- If the owner is not specified, use "Unassigned".
- Include only actionable tasks.
- Do not include general updates.

Meeting Notes:
{meeting_notes}
"""


# =====================================
# 6. Generate Structured Output
# =====================================

try:

    response = client.interactions.create(
        model="gemini-4-hydrogen",
        input=prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": ActionItemList.model_json_schema()
        }
    )
    
    action_data = ActionItemList.model_validate_json(
        response.output_text
    )

    if action_data is None:
        raise ValueError("No structured output received.")

    action_dict = action_data.model_dump()

    print("\nAction items extracted successfully!")


    # =====================================
    # 7. Save TXT File
    # =====================================

    with open(TEXT_OUTPUT, "w", encoding="utf-8") as file:

        for index, item in enumerate(
            action_dict["action_items"], start=1
        ):

            file.write(f"Task {index}\n")
            file.write(f"Task: {item['task']}\n")
            file.write(f"Owner: {item['owner']}\n")
            file.write("-" * 40 + "\n")

    print("Saved:", TEXT_OUTPUT)


    # =====================================
    # 8. Save JSON File
    # =====================================

    with open(JSON_OUTPUT, "w", encoding="utf-8") as file:

        json.dump(
            action_dict,
            file,
            indent=4,
            ensure_ascii=False
        )

    print("Saved:", JSON_OUTPUT)


    # =====================================
    # 9. Load JSON File
    # =====================================

    with open(JSON_OUTPUT, "r", encoding="utf-8") as file:
        loaded_data = json.load(file)


    # =====================================
    # 10. Print Action Items
    # =====================================

    print("\n========== ACTION ITEMS ==========")

    for index, item in enumerate(
        loaded_data["action_items"], start=1
    ):

        print(f"\nTask {index}: {item['task']}")
        print(f"Owner: {item['owner']}")

    print("\nTotal Tasks:", len(loaded_data["action_items"]))


    # =====================================
    # 11. Print Token Usage
    # =====================================

    usage = response.usage

    if usage:

        print("\n========== TOKEN USAGE ==========")
        print("Input Tokens:", usage.total_input_tokens or 0)
        print("Output Tokens:", usage.total_output_tokens or 0)
        print("Thought Tokens:", usage.total_thought_tokens or 0)
        print("Total Tokens:", usage.total_tokens or 0)


except Exception as error:

    print("\nSomething went wrong!")
    print("Error:", error)


Current folder: C:\Users\MD Rihan\Desktop\python-practice-ai\7_file-handiling

Meeting notes loaded successfully!

Action items extracted successfully!
Saved: C:\Users\MD Rihan\Desktop\python-practice-ai\7_file-handiling\action_items.txt
Saved: C:\Users\MD Rihan\Desktop\python-practice-ai\7_file-handiling\action_items.json

========== ACTION ITEMS ==========

Task 1: Check all the machines and finish the machine check by Friday
Owner: Anjali

Task 2: Fix the painting step
Owner: Unassigned

Task 3: Plan some extra shifts to cover the gap
Owner: Ramesh

Task 4: Send a short report on the paint problem to Meena by tomorrow
Owner: Vikram

Total Tasks: 4

========== TOKEN USAGE ==========
Input Tokens: 511
Output Tokens: 105
Thought Tokens: 631
Total Tokens: 1247


In [2]:
from pathlib import Path

print("Jupyter is running from:", Path.cwd())
print("Available files:")

for file in Path.cwd().iterdir():
    print(file.name)

Jupyter is running from: C:\Users\MD Rihan\Desktop\python-practice-ai\7_file-handiling
Available files:
.ipynb_checkpoints
main.ipynb
meeting.txt


In [3]:

from pathlib import Path

meeting_notes = """
Production Meeting - May 29

Ramesh (Production): Last week we made 4,200 units, but our target was 5,000. Machine 3 was down for two days, so we fell behind.

Anjali (Maintenance): Machine 3 is working again now. I will check all the machines this week so we do not get sudden breakdowns.

Vikram (Quality): We found 80 units with paint problems. The paint is not drying properly. We must fix the painting step.

Suresh (Store): The steel supply is late again. The new stock will arrive on Thursday.

Meena (Manager): Okay. Ramesh, please plan some extra shifts to cover the gap. Anjali, finish the machine check by Friday. Vikram, send me a short report on the paint problem by tomorrow.
"""

Path("meeting.txt").write_text(
    meeting_notes.strip(),
    encoding="utf-8"
)

print("Meeting notes saved successfully!")


Meeting notes saved successfully!


In [4]:
with open("meeting.txt", "r", encoding="utf-8") as file:
    meeting_notes = file.read()

print(meeting_notes)
print("\nFile characters:", len(meeting_notes))

Production Meeting - May 29

Ramesh (Production): Last week we made 4,200 units, but our target was 5,000. Machine 3 was down for two days, so we fell behind.

Anjali (Maintenance): Machine 3 is working again now. I will check all the machines this week so we do not get sudden breakdowns.

Vikram (Quality): We found 80 units with paint problems. The paint is not drying properly. We must fix the painting step.

Suresh (Store): The steel supply is late again. The new stock will arrive on Thursday.

Meena (Manager): Okay. Ramesh, please plan some extra shifts to cover the gap. Anjali, finish the machine check by Friday. Vikram, send me a short report on the paint problem by tomorrow.

File characters: 689
